# Practical - 02 - Word2Vec Exploration and Visualisation

### Train a word2vec model on a text corpus using Gensim. Test analogies and find most-similar words.
### Visualize embeddings of 50 words across different domains (sports, food, tech) using t-SNE/PCA.

In [3]:
import os
import sys

# 1. Look up one directory level from 'notebook', then go into 'visualization'
# This ensures it works no matter where you launch the notebook
base_dir = os.path.dirname(os.getcwd())
visualization_dir = os.path.join(base_dir, 'visualization')

if visualization_dir not in sys.path:
    sys.path.insert(0, visualization_dir)

In [4]:
"""Importing the Libraries"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize
import nltk
from gensim.utils import simple_preprocess # lowercases + tokenizes text cleanly
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from dataloader import (                     # our own module inside visualisation/
    load_corpus,          # loads CSV → DataFrame
    preprocess,           # DataFrame → token lists
    get_word_color,       # returns colour per word topic
    filter_vocab_words,   # drops words not in model vocab
    VISUAL_WORDS,         # the 50 target words
    GROUP_COLORS,         # topic → colour mapping
)

In [8]:
"""Load the dataset"""
df = load_corpus("../data/text_corpus.csv")


Loaded 80 sentences from ../data/text_corpus.csv


In [9]:
df.head() # Load first 5 columns

,text
0,The football match ended with a brilliant goal...
1,Cricket is a sport where the bat and ball defi...
2,Tennis requires a coach to improve your techni...
3,The coach trained the player every morning bef...
4,A football player needs speed and stamina to s...


In [10]:
df.shape

(80, 1)

In [11]:
df.info() # checks the detail info of dataset

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    80 non-null     object
dtypes: object(1)
memory usage: 772.0+ bytes


In [12]:
df.isnull().sum()

text    0
dtype: int64

### PreProcess the corpus data

In [13]:
"""Converting raw text rows into a list of token lists — the exact format Word2Vec expects as input."""
sentences = []
for text in df['text']:
    tokens = simple_preprocess(str(text)) # coverts into lowercase and tokenise
    sentences.append(tokens)

In [14]:
# after preprocess into lowercase and tokenising -- Sentence --> word
# eg: Cristiano Ronaldo is the greatest player. ---> ["cristiano", "Ronaldo", "is", "the", "greatest", "player"]

## Train Word2Vec

In [15]:
"""Training the Word2Vec model on preprocessed sentences.Each unique word gets a 100-dimensional vector representation."""

model = Word2Vec(
    sentences=sentences,  # the tokenized list of sentences
    vector_size=100,      # each word becomes a 100-dim vector - Embedding Size
    window=5,             # looks 5 words left and right as context
    min_count=2,          # ignores words that appear less than twice - Min word frequency
    workers=4,            # uses 4 CPU threads to speed up training
    sg=1,                 # sg=1 means Skip-Gram (sg=0 would be CBOW)
    epochs=20             # passes through the entire corpus 20 times
)
print(f"Vocab size: {len(model.wv)}")

Vocab size: 123
